In [2]:
%load_ext autoreload
%autoreload 2

from bot_manager import BotManager
import os
import uuid
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, AIMessage

# 환경 변수 로드
load_dotenv()

# 봇 매니저 초기화 및 봇 선택 (예: bupa)
manager = BotManager()
selected_key = "bupa"
bot = manager.get(selected_key)

# 세션 관리를 위한 고유 ID 생성[cite: 6]
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

print(f"✅ {selected_key.upper()} 봇과 대화를 시작합니다. (Thread ID: {thread_id})")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
🚀 전체 DB 로드 시작...


Loading weights: 100%|██████████| 391/391 [00:01<00:00, 284.18it/s]


✅ bge-m3 로드 완료 (cuda)
✅ Bupa DB 로드 완료
⏳ 모델 및 벡터 DB 로드 중...


C:\skn24\team_project_3\Insurance_Benefit_chatbot\Insurance_Benefit_Chatbot\src\tricare\tricare_core.py:106: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 2665.49it/s]


✅ 텍스트 벡터 DB: 326개 벡터
✅ 표 벡터 DB: 278개 벡터
✅ BM25 인덱스: 326개 청크
✅ 로드 완료
✅ TRICARE DB 로드 완료
✅ Cigna DB 로드 완료
✅ 전체 봇 로드 완료

✅ BUPA 봇과 대화를 시작합니다. (Thread ID: c3a4f201-868d-4426-91d1-3e109aabed32)


In [3]:
def ask_bot(question: str):
    # 1. 이전 대화 상태(State) 가져오기[cite: 2]
    snapshot = bot.get_state(config)
    existing_values = snapshot.values if snapshot.values else {}

    # 2. 봇 실행 (Invoke)[cite: 4]
    result = bot.invoke(
        {
            "messages": [HumanMessage(content=question)],
            "plan_or_intent": existing_values.get("plan_or_intent"),
            "known_treatment": existing_values.get("known_treatment"),
            "slots": existing_values.get("slots", {}),
            "followup_count": existing_values.get("followup_count", 0),
            "max_followups": 2,
            "normalized_query": {},
            "retrieved_docs": [],
            "current_question": "",
            "clarification_msg": "",
            "extra": existing_values.get("extra", {}),
        },
        config=config
    )

    # 3. 답변 및 현재 파악된 정보(State) 출력
    answer = result["messages"][-1].content
    plan = result.get("plan_or_intent", "미파악")
    treatment = result.get("known_treatment", "미파악")
    
    print(f"\n🤖 Bot: {answer}")
    print(f"📊 [상태 파악] 플랜: {plan} | 치료항목: {treatment}")
    print("-" * 50)

In [4]:
print("대화를 종료하려면 '종료' 또는 'exit'를 입력하세요.")

while True:
    user_input = input("👤 User: ")
    print(f"👤 User: {user_input}")
    if user_input.lower() in ['종료', 'exit', 'quit']:
        print("대화를 종료합니다.")
        break
    
    if not user_input.strip():
        continue
        
    ask_bot(user_input)

대화를 종료하려면 '종료' 또는 'exit'를 입력하세요.
👤 User: 보험 추천 좀 해줄래?

🤖 Bot: 죄송하지만, 법적으로 보험 추천은 드릴 수 없습니다. 대신 가입하신 보험의 보장 내용에 대해 안내해 드릴 수 있습니다. 도움이 필요하시면 말씀해 주세요!
📊 [상태 파악] 플랜: None | 치료항목: None
--------------------------------------------------
👤 User: select 플랜이야

🤖 Bot: SELECT 플랜에 대한 정보는 다음과 같습니다.

1. **연간 최대 보장 한도**: SELECT 플랜의 전체 연간 정책 최대 한도는 GBP 1,000,000, EUR 1,250,000, USD 1,700,000입니다. (출처: Bupa-Global-DAC-Select-Global-Health-Plan-MEMBERSHIP-GUIDE-2024.05.pdf, p. 18) 

2. **사전 승인 필요 항목**: 비만 수술, 예방적 수술, 내부 심장 제세동기, 재건 수술, 재활, 암 치료, 고급 치료 의약품(ATMPs), 교통(대피), 5일 이상의 모든 입원 치료에 대해 사전 승인이 필요합니다. (출처: Bupa-Global-DAC-Select-Global-Health-Plan-MEMBERSHIP-GUIDE-2024.05.pdf, p. 18)

3. **외래 진료 한도**: 외래 진료는 연간 최대 GBP 7,500, EUR 9,400 또는 USD 12,800까지 전액 지급됩니다. (출처: Bupa-Global-DAC-Select-Global-Health-Plan-MEMBERSHIP-GUIDE-2024.05.pdf, p. 18)

4. **공동 부담금**: 필수 15% 및 선택적 25%의 공동 부담금이 적용됩니다. (출처: Bupa-Global-DAC-Select-Global-Health-Plan-MEMBERSHIP-GUIDE-2024.05.pdf, p. 18)

이 정보가 도움이 되길 바랍니다. 추가 질문이 있